# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!


**Approach / notes:** [week1-exercise-approach.md](week1-exercise-approach.md)

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:3b'

In [ ]:
# set up environment - OPENAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OPENAI API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = MODEL_GPT
openai = OpenAI()

OPENAI API key looks good so far


In [ ]:
# set up environment - OLLAMA
OLLAMA_BASE_URL = "http://localhost:11434/v1"
    
# MODEL = MODEL_LLAMA
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama_api_key")

In [26]:
# here is the question; type over this to ask something new

# question = """
# Please explain what this code does and why:
# yield from {book.get("author") for book in books if book.get("author")}
# """

question = """
What is the difference between AWS Lambda provisioned concurrency and standard concurrency, 
and how do they impact cold starts?
"""


In [27]:
# Ollama prompts
ollama_system_prompt="""
Provide a raw, technically precise, and concise explanation of the mechanics. 
Include exact parameters or code if applicable. No fluff.
"""

ollama_user_prompt=f"""
Here is the question:
{question}
"""                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
# Get Llama 3.2 to answer
ollama_response=ollama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = [
        {"role":"system", "content":ollama_system_prompt},
        {"role":"user", "content": ollama_user_prompt}
        ],
)

ollama_result = ollama_response.choices[0].message.content
# display(Markdown(ollama_result))

In [ ]:
# GPT prompts
openai_system_prompt = """
You are an elite technical educator. You will be given a user's original question and a raw, 
highly technical response from a local model. Your job is to transform this raw data into 
a masterfully structured study guide. Always include: 1) A simple real-world analogy, 2) 
A deeply technical breakdown structured with clear scannable headers, and 3) A summary 
checklist or configuration takeaway. Maintain high technical density—do not sacrifice 
accuracy for simplicity.
"""

openai_user_prompt = f"""
Here is the original question - "{question}"
 
Here is the raw output of the local model. Respond with a clear and well structure explaination.
{ollama_response}
"""

In [25]:
# Get gpt-4o-mini to answer, with streaming
openai_stream=openai.chat.completions.create(
    model = MODEL_GPT,
    messages=[
        {"role":"system", "content":openai_system_prompt},
        {"role":"user", "content":openai_user_prompt}
        ],
    stream=True
)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in openai_stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


# Study Guide: Creating a Custom Model for Electricity Regulatory Information Analytics

## 1) Real-World Analogy
Imagine you're setting up a personal library to track your collection of historical novels. You need to sort the books (data preprocessing), organize them by themes like "love," "adventure," and "mystery" (feature engineering), choose which books best reflect the genre you love (model selection), and finally, ask your friends for their opinions on your choices (model evaluation). Just like managing a library, building a model involves careful organization, choice-making, and feedback loops to improve your collection (model performance).

---

## 2) Technical Breakdown

### Step 1: Data Preprocessing

#### Import Necessary Libraries
Utilize the following libraries for data operations:
- **Pandas**: Data manipulation and analysis.
- **NumPy**: For numerical computations.
- **Scikit-learn**: For implementing machine learning algorithms.

#### Load Tariff Orders Data
Load your dataset with:
```python
import pandas as pd
import numpy as np

# Load CSV file containing tariff orders data
df = pd.read_csv('tariff_orders.csv')
```

#### Clean and Preprocess Data
Transform and clean your dataset:
```python
# Convert date columns to datetime format
df['date'] = pd.to_datetime(df['date'])

# Extract relevant information from the data (e.g., price, volume, etc.)
df[['price', 'volume']] = pd.melt(df, id_vars=['date'], var_name='meter_type', value_name='value')

# Handle missing values and outliers
df.dropna(inplace=True)
```

### Step 2: Feature Engineering

#### Extract Relevant Features
Create new variables to enhance your dataset:
```python
# Price trends (e.g., slope, intercept)
df['price_trend'] = np.polyfit(df['date'].astype(int), df['value'].values, 1)[0]

# Volume patterns (e.g., standard deviation, mean)
df['volume_pattern'] = df['value'].rolling(window=10).std()
```

#### Encode Categorical Variables
Transform categorical data into numerical format:
```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['meter_type_encoded'] = le.fit_transform(df['meter_type'])
```

### Step 3: Model Selection and Training

#### Choose a Machine Learning Algorithm
Select the algorithm based on your problem context. Examples include:
- **Linear Regression** 
- **Decision Trees**
- **Random Forest**

#### Training the Model
Train your model with the prepared data:
```python
from sklearn.linear_model import LinearRegression

# Train a model using Linear Regression as an example
model = LinearRegression()
model.fit(df.drop('target_variable', axis=1), df['target_variable'])
```

### Step 4: Model Tuning and Evaluation

#### Hyperparameter Optimization
Optimize model performance using techniques like grid search:
```python
from sklearn.model_selection import GridSearchCV

param_grid = {'verbose': [0, 1], 'n_estimators': range(10, 100, 10)}
grid_search = GridSearchCV(model, param_grid, cv=5)
grid_search.fit(df.drop('target_variable', axis=1), df['target_variable'])
```

#### Evaluate the Model
Assess model accuracy with performance metrics:
```python
from sklearn.metrics import mean_absolute_error

# Generate predictions
predictions = model.predict(df.drop('target_variable', axis=1))

# Calculate and print MAE
mae = mean_absolute_error(df['target_variable'], predictions)
print(f'MAE: {mae:.3f}')
```

### Example Use Cases
- Predicting electricity demand using historical tariff orders data.
- Analyzing price and volume trends to forecast future energy prices.

---

## 3) Summary Checklist / Configuration Takeaway
- **Data Loading**: Ensure your data is correctly imported as a DataFrame.
- **Data Cleaning**: Convert data types, handle nulls and outliers, and ensure tidy format.
- **Feature Engineering**: Identify critical features that influence tariff outcomes and encode categorical data appropriately.
- **Model Selection**: Choose a model suited to your analytical goals (e.g., regression vs. classification).
- **Model Training**: Fit the model to your processed data without the target variable.
- **Hyperparameter Tuning**: Use techniques like GridSearchCV to improve model performance.
- **Model Evaluation**: Employ metrics such as MAE and RMSE to provide quantitative performance assessments. 

Adapt these elements based on your specific dataset and problem statement for optimal outcomes in electricity regulatory information analytics.